In [ ]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Model
import numpy as np



In [ ]:
df = pd.read_csv("/content/arabic_english_dataset.csv")

df.head()

,english,arabic
0,hello,مرحبا
1,good morning,صباح الخير
2,good evening,مساء الخير
3,how are you,كيف حالك
4,I am fine,أنا بخير


In [ ]:
df["arabic"] = df["arabic"].astype(str)

df["decoder_input"] = "<start> " + df["arabic"]
df["decoder_target"] = df["arabic"] + " <end>"

In [ ]:
eng_tokenizer = Tokenizer(filters="")
eng_tokenizer.fit_on_texts(df["english"])

encoder_sequences = eng_tokenizer.texts_to_sequences(df["english"])

eng_vocab_size = len(eng_tokenizer.word_index) + 1

In [ ]:
ar_tokenizer = Tokenizer(filters="")

ar_tokenizer.fit_on_texts(
    list(df["decoder_input"]) +
    list(df["decoder_target"])
)

decoder_input_sequences = ar_tokenizer.texts_to_sequences(df["decoder_input"])

decoder_target_sequences = ar_tokenizer.texts_to_sequences(df["decoder_target"])

ar_vocab_size = len(ar_tokenizer.word_index) + 1

In [ ]:
max_encoder_len = max(len(x) for x in encoder_sequences)
max_decoder_len = max(len(x) for x in decoder_input_sequences)

encoder_input = pad_sequences(
    encoder_sequences,
    maxlen=max_encoder_len,
    padding="post"
)

decoder_input = pad_sequences(
    decoder_input_sequences,
    maxlen=max_decoder_len,
    padding="post"
)

decoder_target = pad_sequences(
    decoder_target_sequences,
    maxlen=max_decoder_len,
    padding="post"
)

In [ ]:
latent_dim = 64

#encoder
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(eng_vocab_size, 64)(encoder_inputs)

encoder_lstm = LSTM(
    latent_dim,
    return_state=True
)

encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)

encoder_states = [state_h, state_c]

#decoder
decoder_inputs = Input(shape=(None,))
dec_emb_layer = Embedding(ar_vocab_size, 64)
dec_emb = dec_emb_layer(decoder_inputs)

decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True
)

decoder_outputs, _, _ = decoder_lstm(
    dec_emb,
    initial_state=encoder_states
)

decoder_dense = Dense(
    ar_vocab_size,
    activation="softmax"
)

decoder_outputs = decoder_dense(decoder_outputs)

In [ ]:

model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, None, 64)  │      5,184 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, None, 64)  │      5,056 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, 64),      │     33,024 │ embedding_2[0][0] │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ [(None, None,     │     33,024 │ embedding_3[0][0… │
│                     │ 64), (None, 64),  │            │ lstm_2[0][1],     │
│                     │ (None, 64)]       │            │ lstm_2[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None, 79)  │      5,135 │ lstm_3[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 81,423 (318.06 KB)

 Trainable params: 81,423 (318.06 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
decoder_target = np.expand_dims(decoder_target, -1)

In [ ]:
model.fit(
    [encoder_input, decoder_input],
    decoder_target,
    batch_size=8,
    epochs=20,
    validation_split=0.2
)

Epoch 1/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 127ms/step - accuracy: 0.2063 - loss: 4.3589 - val_accuracy: 0.3750 - val_loss: 4.3364
Epoch 2/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.3125 - loss: 4.3208 - val_accuracy: 0.3750 - val_loss: 4.2943
Epoch 3/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.3125 - loss: 4.2730 - val_accuracy: 0.3750 - val_loss: 4.2265
Epoch 4/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.3125 - loss: 4.1900 - val_accuracy: 0.3750 - val_loss: 4.1104
Epoch 5/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.3125 - loss: 4.0462 - val_accuracy: 0.3750 - val_loss: 3.8968
Epoch 6/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.3125 - loss: 3.7609 - val_accuracy: 0.3750 - val_loss: 3.5098
Epoch 7/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.3125 - loss: 3.3221 - val_accuracy: 0.3750 - val_loss: 2.9719
Epoch 8/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.3125 - loss: 2.9111 - val_accuracy: 0.3750 - val_loss: 2.6729